In [25]:
import json
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import numpy as np

ROOT = Path.cwd()
json_path = ROOT / "cw19-03-json.json"

In [19]:
def metadata_func(record,metadata):
    metadata["doc_id"] = record["doc_id"]
    metadata["source"] = record["source"]
    metadata["access_level"] = record["access_level"]
    metadata["version_status"] = record["version_status"]

    return metadata

In [20]:
def create_splitter(chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return text_splitter

In [21]:
loader = JSONLoader(
    file_path=json_path,
    metadata_func=metadata_func,
    jq_schema=".[]",
    content_key="text",

)

docs = loader.load()

len_docs = []
for doc in docs:
    len_docs.append(len(doc.page_content))

percents = [25, 50, 75,90]
percentiles = np.percentile(len_docs, percents)
for i,per in enumerate(percentiles):
    print(f"{percents[i]}% {per}")


25% 488.25
50% 500.0
75% 629.5
90% 757.0


In [22]:
len_docs

[498, 485, 842, 672, 444, 502]

In [23]:
int_percentiles = [int(x) for x in percentiles]
print(int_percentiles)
splitters = [create_splitter(chunk_size=perc,chunk_overlap=int(perc*0.05)) for perc in int_percentiles]

[488, 500, 629, 757]


# percentile

In [24]:
loader = JSONLoader(
    file_path=json_path,
    metadata_func=metadata_func,
    jq_schema=".[]",
    content_key="text",

)
percentiles_docs = dict()
for i,splitter in enumerate(splitters):


    docs = loader.load_and_split(text_splitter=splitter)
    print(f"percentile {int_percentiles[i]}, overlap size {int_percentiles[i]*0.05:.0f}")
    print(f"len third content {len(docs[2].page_content)}")
    print(f"slice of content \n{docs[2].page_content}")
    print("---"*10)
    print(f"metadata {docs[2].metadata}")
    print(f"sum chunks {len(docs)}")
    print("###"*10)
    percentiles_docs[f"{percents[i]}"] = docs

percentile 488, overlap size 24
len third content 485
slice of content 
تاریخ انتشار: مهر ۱۴۰۵

جلسه ۱۷ به مقدمه RAG اختصاص دارد و ساعت ۹:۰۰ آغاز می‌شود. در این جلسه، نقش اطلاعات بیرونی در پاسخ‌دادن مدل‌های زبانی بررسی خواهد شد.

جلسه ۱۸ با موضوع Retrieval ساعت ۱۴:۰۰ برگزار می‌شود. بخش اصلی آن درباره جست‌وجو در اسناد، رتبه‌بندی متن‌های مرتبط و بررسی نتیجه بازیابی است.

جلسه ۱۹ با موضوع ارزیابی RAG ساعت ۱۶:۰۰ برگزار می‌شود. دانشجوها در این جلسه درباره معیارهای بازیابی و بررسی کیفیت شواهد صحبت می‌کنند.

این تقویم برای بازه آموزشی مهر ۱۴۰۵ تنظیم شده است.
------------------------------
metadata {'source': 'schedule_v2.md', 'seq_num': 2, 'doc_id': 'DOC-02', 'access_level': 'public', 'version_status': 'current'}
sum chunks 10
##############################
percentile 500, overlap size 25
len third content 422
slice of content 
ثبت غیبت‌ها

حضور دانشجویان در پایان هر جلسه ثبت می‌شود. دانشجو می‌تواند وضعیت حضور خود را از مسئول دوره پیگیری کند.

اگر به دلیل بیماری یا شرایط خاص در جلسه‌ای غایب ب

# Vectorization

In [29]:
docs_vectorized = []

query = "قوانین مربوط به غیبت در دوره "
query_v = []


for doc  in percentiles_docs:
    tf_idf = TfidfVectorizer()

    text = [doc.page_content for doc in docs]
    docs_vectorized.append(tf_idf.fit_transform(text))
    query_v.append(tf_idf.transform([query]))

In [33]:
query = "قوانین مربوط به غیبت در دوره "

tf_idf = TfidfVectorizer()
tf_idf.fit_transform([query]).toarray()

array([[0.40824829, 0.40824829, 0.40824829, 0.40824829, 0.40824829,
        0.40824829]])